<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_080_Modelos_de_lenguaje_funcionamiento_limitaciones_y_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 80. Fundamentos de los modelos de lenguaje de gran tamaño: funcionamiento, limitaciones y uso seguro mediante API</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 2. Programando con IA**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Retos en la app:** [las tres misiones del Tema 80](https://fborrasumh.github.io/bioestadistica/#tema-80)

**Requisito:** tener una clave de acceso guardada en Secretos (Tema 10, ampliación). Sin clave, el cuaderno funciona con respuestas de ejemplo.

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

**Para dar la clase con el cuaderno:** antes de empezar, usa **Entorno de ejecución → Ejecutar todo**. Así se dibujan los gráficos. Las celdas con borde azul son las diapositivas. Debajo de cada una está el desarrollo del bloque; pliégalo con la flecha de su título. El índice (☰, a la izquierda) sirve para saltar de un bloque a otro.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Explicar cómo genera texto un modelo de lenguaje de gran tamaño (LLM): tokens y predicción probabilística de la siguiente palabra.
2. Describir el efecto de la **temperatura** sobre las respuestas.
3. Identificar las principales limitaciones de los LLM: alucinaciones, fecha de corte, variabilidad y sesgos.
4. Enviar una petición a un modelo mediante su **API** desde Python, con la clave guardada en Secretos.
5. Pedir una respuesta en formato JSON y comprobarla antes de usarla.
6. Calcular el consumo de tokens de una petición.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: de la probabilidad a la API | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Cinco referencias con DOI… y dos no existen</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_080_referencias.png" width="780" alt="Cinco referencias con DOI… y dos no existen"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>El modelo genera el texto <b>más probable</b>; una referencia plausible lo es, exista o no.</li><li>Todo lo que afirme un modelo de lenguaje se verifica en la fuente.</li></ul><p style="font-size:21px;margin:14px 0 0 0;color:#8A4B08"><b>Pregunta para la clase:</b> ¿Cómo comprobaríais en dos minutos si una referencia existe?</p></div>

#### Desarrollo para el estudio

Un estudiante de sexto curso prepara una sesión clínica sobre el tratamiento de la hiponatremia. Pide a un asistente de IA «las cinco referencias más importantes, con su DOI». El asistente responde al instante con cinco artículos bien formateados: autores, revista, año, volumen y DOI.

Al buscarlos, dos no existen. Los autores son reales, las revistas también, pero esos artículos nunca se publicaron. Un tercero existe, pero no dice lo que el asistente afirma.

El asistente no ha «mentido». Ha hecho exactamente aquello para lo que está construido: **generar el texto más probable** después de la pregunta. Una referencia bibliográfica plausible es un texto muy probable, exista o no. Entender cómo funciona un modelo de lenguaje es la mejor protección frente a sus errores.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 Cómo genera texto un LLM</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Un LLM predice la siguiente palabra</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_080_siguiente.png" width="780" alt="Un LLM predice la siguiente palabra"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>El modelo asigna una probabilidad a cada palabra posible y elige una.</li><li>Repite el proceso palabra a palabra: no consulta una base de datos de hechos.</li></ul></div>

#### Desarrollo para el estudio

Un **modelo de lenguaje de gran tamaño** (LLM, *large language model*) es una red neuronal entrenada con enormes cantidades de texto. Su tarea parece simple: **predecir el siguiente fragmento de texto**.

1. El texto se divide en **tokens**: palabras, partes de palabras o signos. En español, una palabra equivale de media a algo más de un token.
2. Para cada posición, el modelo calcula una **distribución de probabilidad** sobre todos los tokens posibles.
3. Se **elige** un token según esas probabilidades, se añade al texto y se repite el proceso.

Así, tras «El paciente refiere dolor torácico de características…», el token «opresivas» tendrá una probabilidad alta, y «azules», una muy baja. El modelo no consulta una base de datos ni comprueba hechos: **genera lo probable**.

### <font color='steelblue'>2.2 La temperatura</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">La temperatura reparte la probabilidad</p><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Temperatura baja: casi siempre la palabra más probable (respuestas estables).</li><li>Temperatura alta: más variedad y más riesgo de incoherencias.</li><li>Ninguna temperatura garantiza que la respuesta sea verdadera.</li></ul></div>

In [ ]:
#@title Figura: La temperatura reparte la probabilidad
import numpy as np
import matplotlib.pyplot as plt

palabras = ["sodio\nhipertónico", "restricción\nhídrica", "tolvaptán", "diuréticos", "esperar"]
logits = np.array([3.0, 2.2, 1.2, 0.6, -0.5])
fig, ejes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for eje, T in zip(ejes, [0.2, 1.0, 2.0]):
    p = np.exp(logits / T) / np.exp(logits / T).sum()
    eje.bar(range(5), p, color=["firebrick"] + ["steelblue"] * 4)
    eje.set_xticks(range(5), palabras, fontsize=9)
    eje.set_title(f"Temperatura = {T}", fontweight="bold")
ejes[0].set_ylabel("Probabilidad de la siguiente palabra")
plt.suptitle("«El tratamiento de la hiponatremia grave sintomática es…»", fontsize=13)
plt.tight_layout()
plt.show()

#### Desarrollo para el estudio

La **temperatura** controla cómo se elige el siguiente token:

| Temperatura | Efecto | Uso |
| --- | --- | --- |
| Baja (cercana a 0) | Casi siempre elige el token más probable: respuestas estables y repetitivas | Extraer datos, clasificar, generar código |
| Media (alrededor de 1) | Respeta las probabilidades del modelo | Uso general |
| Alta (mayor de 1) | Da más oportunidades a tokens poco probables: más variedad y más errores | Generar ideas |

Por eso **la misma pregunta puede recibir respuestas distintas**. En el Tema 190 mediremos esa variabilidad con métodos estadísticos.

### <font color='steelblue'>2.3 Limitaciones</font>

<img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_080_esquema.png" width="760" alt="Seis limitaciones (alucinación, fecha de corte, variabilidad, sesgos, complacencia, cálculo poco fiable) llevan a una regla: toda respuesta se verifica.">

| Limitación | Qué significa | Ejemplo clínico |
| --- | --- | --- |
| **Alucinación** | Genera información falsa con aspecto convincente | Referencias, dosis o estudios inexistentes |
| **Fecha de corte** | Solo conoce lo que había en sus datos de entrenamiento | Desconoce una guía clínica publicada después |
| **Variabilidad** | La misma pregunta da respuestas distintas | Dos explicaciones contradictorias de una prueba |
| **Sesgos** | Reproduce los sesgos de los textos con que se entrenó | Peor rendimiento en poblaciones infrarrepresentadas |
| **Complacencia** | Tiende a dar la razón a quien pregunta | Confirma un diagnóstico erróneo sugerido en la pregunta |
| **Cálculo poco fiable** | Predice texto, no calcula | Errores en operaciones con varios pasos |

La regla de la asignatura se deriva de todo esto: **toda respuesta se verifica**.

### <font color='steelblue'>2.4 Qué es una API</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Una API: tu código habla con el modelo</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_080_api.png" width="780" alt="Una API: tu código habla con el modelo"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Tu programa envía una petición (prompt + clave) y recibe la respuesta en JSON.</li><li>Cada llamada cuesta dinero o cuota, y lo que envías sale de tu ordenador: nunca datos de pacientes.</li></ul></div>

#### Desarrollo para el estudio

Una **API** (*application programming interface*) permite que un programa use un servicio de otro. Usar un LLM por su API consiste en:

1. Enviar una **petición** con la pregunta (*prompt*), el modelo elegido y parámetros como la temperatura.
2. Identificarse con la **clave de acceso** (API key).
3. Recibir una **respuesta** en JSON (Tema 70) con el texto generado y el número de tokens consumidos.

El coste se mide en **tokens** de entrada (la pregunta) y de salida (la respuesta). En el nivel gratuito no se paga, pero hay un límite de peticiones por minuto y por día.

## <font color='steelblue'>3. Demostración: de la probabilidad a la API (20 min)</font>

### <font color='steelblue'>3.1 Un modelo de juguete: elegir la siguiente palabra</font>

Simulamos las puntuaciones que un modelo podría asignar a cinco continuaciones posibles de «El paciente refiere dolor torácico de características…». La función **softmax** convierte las puntuaciones en probabilidades, y la temperatura las divide antes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

candidatos = ["opresivas", "punzantes", "pleuríticas", "mecánicas", "azules"]
puntuaciones = np.array([4.0, 2.5, 2.0, 1.5, -2.0])


def probabilidades(puntuaciones, temperatura):
    """Softmax con temperatura: convierte puntuaciones en probabilidades."""
    z = puntuaciones / temperatura
    exp_z = np.exp(z - z.max())
    return exp_z / exp_z.sum()


fig, ejes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for eje, t in zip(ejes, [0.2, 1.0, 2.0]):
    eje.bar(candidatos, probabilidades(puntuaciones, t), color="steelblue")
    eje.set_title(f"Temperatura {t}")
    eje.tick_params(axis="x", rotation=45)
ejes[0].set_ylabel("Probabilidad")
plt.tight_layout()
plt.show()

Con temperatura 0,2, «opresivas» acapara casi toda la probabilidad. Con temperatura 2, incluso «azules» tiene alguna posibilidad. Generamos 10 continuaciones con cada temperatura:

In [ ]:
rng = np.random.default_rng(seed=80)
for t in [0.2, 1.0, 2.0]:
    elegidas = rng.choice(candidatos, size=10, p=probabilidades(puntuaciones, t))
    print(f"T = {t}: {', '.join(elegidas)}")

### <font color='steelblue'>3.2 Preparar la conexión con la API</font>

Usaremos la API de **Google Gemini** con la biblioteca oficial `google-genai`. La celda lee la clave de Secretos **sin mostrarla**. Si no hay clave, el resto del cuaderno usará respuestas de ejemplo.

El nombre del modelo cambia con el tiempo. Si la petición falla porque el modelo no existe, ejecuta la celda de ampliación «Ver los modelos disponibles». Después, cambia la variable `MODELO`.

In [ ]:
%pip install -q google-genai

MODELO = "gemini-2.5-flash"

try:
    from google.colab import userdata
    from google import genai
    from google.genai import types
    cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    print("Conexión preparada con el modelo", MODELO)
except Exception as error:
    cliente = None
    print("Sin clave o fuera de Colab: se usarán respuestas de ejemplo.", type(error).__name__)

### <font color='steelblue'>3.3 Una función para preguntar</font>

Encapsulamos la petición en una función (Tema 50). Devuelve el texto y el número de tokens consumidos.

In [ ]:
def preguntar(prompt, temperatura=0.2):
    """Envía un prompt al modelo y devuelve (texto, tokens_entrada, tokens_salida)."""
    if cliente is None:
        return ("[Sin clave configurada: aquí aparecería la respuesta del modelo]",
                len(prompt.split()), 0)
    respuesta = cliente.models.generate_content(
        model=MODELO,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temperatura),
    )
    uso = respuesta.usage_metadata
    return respuesta.text, uso.prompt_token_count, uso.candidates_token_count


texto, entrada, salida = preguntar(
    "Explica en dos frases cómo se calcula la presión arterial media a partir de la "
    "presión sistólica y la diastólica."
)
print(texto)
print(f"\nTokens: {entrada} de entrada y {salida} de salida")

**Verifica:** ¿coincide la fórmula que da el modelo con la del Tema 10? Compruébalo siempre.

### <font color='steelblue'>3.4 Pedir una respuesta en JSON y comprobarla</font>

Para usar la respuesta en un programa, se pide en un formato estructurado. Después se **comprueba** con un caso de resultado conocido, como en el Tema 50. Aquí pedimos una clasificación cuyo resultado sabemos de antemano.

In [ ]:
import json

prompt = (
    "Clasifica la presión arterial 148/92 mmHg según la guía ESH 2023. "
    'Responde solo con un JSON con las claves "categoria" y "motivo", sin texto adicional.'
)

if cliente is None:
    texto = '{"categoria": "hipertensión grado 1", "motivo": "PAS 140-159 o PAD 90-99"}'
else:
    texto, _, _ = preguntar(prompt, temperatura=0)

texto = texto.strip().removeprefix("```json").removesuffix("```").strip()
datos = json.loads(texto)
print(datos)

assert "grado 1" in datos["categoria"].lower(), "La clasificación no coincide con la guía"
print("Respuesta comprobada")

Las dos líneas `removeprefix`/`removesuffix` limpian una costumbre de muchos modelos: envolver el JSON en marcas de código.

### <font color='steelblue'>3.5 La misma pregunta, varias respuestas</font>

Con temperatura alta, repetimos cinco veces la misma pregunta. Si no hay clave, se simulan respuestas para mostrar el efecto.

In [ ]:
pregunta = "En una sola palabra, ¿cuál es el síntoma más frecuente de la insuficiencia cardiaca?"

for intento in range(1, 6):
    if cliente is None:
        texto = rng.choice(["Disnea", "Disnea", "Disnea", "Fatiga", "Edemas"])
    else:
        texto, _, _ = preguntar(pregunta, temperatura=1.5)
    print(f"Intento {intento}: {texto.strip()}")

### <font color='steelblue'>3.6 Detectar una alucinación</font>

Pide al modelo referencias bibliográficas y **comprueba cada una**: busca el DOI en [doi.org](https://doi.org) o el título en PubMed. Anota cuántas existen y dicen lo que el modelo afirma.

In [ ]:
texto, _, _ = preguntar(
    "Dame tres referencias bibliográficas, con su DOI, sobre la ecuación CKD-EPI 2021 "
    "para estimar el filtrado glomerular.",
    temperatura=0.7,
)
print(texto)
print("\n→ Verifica cada DOI en https://doi.org antes de usar ninguna referencia.")

### <font color='steelblue'>Ampliación: ver los modelos disponibles</font>

In [ ]:
if cliente is not None:
    for modelo in cliente.models.list():
        if "gemini" in modelo.name:
            print(modelo.name)
else:
    print("Necesitas una clave configurada para listar los modelos.")

### <font color='steelblue'>Ampliación: la misma petición con OpenAI</font>

Si tienes una clave de OpenAI guardada como `OPENAI_API_KEY`, la petición equivalente es esta. Comprueba en la documentación de OpenAI el nombre de los modelos disponibles.

In [ ]:
%pip install -q openai

try:
    from openai import OpenAI
    cliente_openai = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
    respuesta = cliente_openai.responses.create(
        model="gpt-4.1-mini",
        input="Explica en una frase qué es la presión arterial media.",
    )
    print(respuesta.output_text)
except Exception as error:
    print("No se ha podido usar OpenAI:", type(error).__name__)

### <font color='steelblue'>Ampliación: buenas prácticas para escribir un prompt</font>

| Práctica | Ejemplo |
| --- | --- |
| Dar contexto y rol | «Eres profesor de bioestadística de un grado de Medicina…» |
| Ser específico | «En tres frases», «para estudiantes de segundo curso» |
| Pedir el formato | «Responde en JSON con las claves…» |
| Pedir el razonamiento | «Explica cada paso del cálculo» |
| Pedir que reconozca sus límites | «Si no estás seguro, dilo» |
| No sugerir la respuesta | Evitar «¿Verdad que es una neumonía?» |

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Escribir la clave en el código** en lugar de leerla de Secretos.

**Confiar en la respuesta sin verificarla**, sobre todo en cifras, dosis y referencias.

**Enviar datos de pacientes**, aunque parezcan anónimos: edad, fecha y diagnóstico raro pueden bastar para identificar a una persona.

**Ignorar los límites del nivel gratuito:** si se superan, la API devuelve un error de cuota (código 429). Basta con esperar.

**Usar un modelo retirado:** los nombres de los modelos cambian; consulta la lista de modelos disponibles.

**Analizar el JSON sin limpiarlo** cuando el modelo lo envuelve en marcas de código.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 80 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** predecir el efecto de la temperatura sobre distribuciones de probabilidad de juguete.
2. **Misión clínica (con pistas):** pedir al modelo que clasifique cinco valores de IMC en JSON. Comprobar cada clasificación con la función del Tema 50.
3. **Misión reto (con asistente):** pedir tres referencias sobre un tema clínico, verificar cada DOI y calcular la proporción de referencias correctas.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': '¿Qué hace un modelo de lenguaje al generar una respuesta?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Elige, token a token, una continuación según su probabilidad',
               'correct': True,
               'feedback': 'Correcto: genera el texto más probable dado lo anterior.'},
              {'answer': 'Busca la respuesta en una base de datos médica',
               'correct': False,
               'feedback': 'No consulta bases de datos: genera texto a partir de lo aprendido.'},
              {'answer': 'Razona como un médico y comprueba los hechos',
               'correct': False,
               'feedback': 'No comprueba hechos; por eso puede alucinar.'},
              {'answer': 'Copia literalmente textos de su entrenamiento',
               'correct': False,
               'feedback': 'Genera texto nuevo a partir de patrones aprendidos.'}]},
 {'question': 'Quieres que el modelo extraiga datos de un texto siempre de la misma forma. ¿Qué '
              'temperatura conviene?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Baja, cercana a 0',
               'correct': True,
               'feedback': 'Correcto: favorece la opción más probable y da respuestas estables.'},
              {'answer': 'Alta, mayor de 1',
               'correct': False,
               'feedback': 'Una temperatura alta aumenta la variedad y los errores.'},
              {'answer': 'Da igual: la temperatura no afecta a la respuesta',
               'correct': False,
               'feedback': 'La temperatura cambia cómo se elige cada token.'},
              {'answer': 'Temperatura negativa',
               'correct': False,
               'feedback': 'La temperatura es un valor positivo.'}]},
 {'question': 'Un modelo te da una referencia con un DOI que no existe. ¿Cómo se llama este '
              'fenómeno?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Alucinación',
               'correct': True,
               'feedback': 'Correcto: información falsa con aspecto convincente.'},
              {'answer': 'Fecha de corte',
               'correct': False,
               'feedback': 'La fecha de corte explica que desconozca lo reciente, no que invente.'},
              {'answer': 'Sesgo de asignación',
               'correct': False,
               'feedback': 'Es un concepto de los ensayos clínicos (Tema 60).'},
              {'answer': 'Error de cuota',
               'correct': False,
               'feedback': 'Es un error de la API por exceso de peticiones.'}]},
 {'question': '¿Dónde debe estar la clave de acceso al usar la API desde Colab?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'En el panel Secretos, leída con userdata.get()',
               'correct': True,
               'feedback': 'Correcto: así no aparece nunca en el código ni en pantalla.'},
              {'answer': 'En una variable al principio del cuaderno',
               'correct': False,
               'feedback': 'Cualquiera que vea el cuaderno vería la clave.'},
              {'answer': 'Dentro del prompt',
               'correct': False,
               'feedback': 'La clave nunca se envía como texto de la pregunta.'},
              {'answer': 'En un fichero CSV junto a los datos',
               'correct': False,
               'feedback': 'Quedaría expuesta al compartir el fichero.'}]},
 {'question': 'Una petición consume 120 tokens de entrada y 380 de salida. ¿Cuántos tokens consume '
              'en total?',
  'type': 'multiple_choice',
  'answers': [{'answer': '500',
               'correct': True,
               'feedback': 'Correcto: el consumo es la suma de entrada y salida.'},
              {'answer': '380',
               'correct': False,
               'feedback': 'También cuentan los tokens de entrada.'},
              {'answer': '260', 'correct': False, 'feedback': 'No se restan: se suman.'},
              {'answer': '120',
               'correct': False,
               'feedback': 'También cuentan los tokens de salida.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- Un LLM genera texto prediciendo, token a token, la continuación más probable; no comprueba hechos.
- La temperatura regula la variabilidad: baja para tareas que exigen estabilidad.
- Alucinaciones, fecha de corte, variabilidad, sesgos y complacencia obligan a verificar toda respuesta.
- La API se usa con la clave guardada en Secretos; la respuesta llega en JSON con el texto y los tokens consumidos.
- Una respuesta en JSON se comprueba con casos conocidos antes de usarla.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| LLM | Modelo de lenguaje de gran tamaño, entrenado para predecir texto |
| Token | Fragmento de texto con el que trabaja el modelo |
| *Prompt* | Instrucción o pregunta que se envía al modelo |
| Temperatura | Parámetro que regula la aleatoriedad en la elección de tokens |
| Softmax | Función que convierte puntuaciones en probabilidades que suman 1 |
| Alucinación | Información falsa generada con apariencia verosímil |
| Fecha de corte | Fecha hasta la que llegan los datos de entrenamiento del modelo |
| API | Interfaz que permite a un programa usar un servicio externo |
| Cuota | Límite de peticiones o tokens permitidos en un periodo |

## <font color='steelblue'>Para saber más</font>

- [Documentación de la API de Gemini](https://ai.google.dev/gemini-api/docs)
- [Documentación de la API de OpenAI](https://platform.openai.com/docs)
- Thirunavukarasu AJ, *et al.* Large language models in medicine. *Nat Med*. 2023;29(8):1930-1940. doi:[10.1038/s41591-023-02448-8](https://doi.org/10.1038/s41591-023-02448-8)

## <font color='steelblue'>Retos de este tema en la app</font>

Practica lo que has visto con las tres misiones del Tema 80 en la app **Bioestadística**. Cuentan para la evaluación continua.

1. **Básica (sin ayuda):** La temperatura y las distribuciones de probabilidad
2. **Clínica (con pistas):** Verificar las clasificaciones de IMC que devuelve un modelo
3. **Reto (con asistente de IA):** Verificar las referencias que propone un modelo

<a href="https://fborrasumh.github.io/bioestadistica/#tema-80" target="_blank" style="display:inline-block;padding:10px 18px;background:#C8461F;color:#fff;border-radius:8px;font-weight:bold;text-decoration:none">Abrir los retos del Tema 80 en la app ↗</a>

---
*Tema 80 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*